In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# ====== 步骤二/三：编写Sigmoid函数（支持矩阵输入） ======
def sigmoid(z):
    """
    计算Sigmoid函数值，支持标量、向量和矩阵输入
    
    参数:
    z -- 输入值，可以是标量、向量或矩阵
    数值稳定的sigmoid函数实现
    返回:
    与输入相同维度的Sigmoid计算结果
    """
    # 对于正数输入，使用负数形式计算避免溢出
    pos_mask = (z >= 0)
    neg_mask = (z < 0)
    
    # 初始化结果数组
    result = np.zeros_like(z)
    
    # 对于正数输入使用稳定的计算方式
    result[pos_mask] = 1 / (1 + np.exp(-z[pos_mask]))
    
    # 对于负数输入，使用另一种形式计算
    exp_z = np.exp(z[neg_mask])
    result[neg_mask] = exp_z / (1 + exp_z)
    
    # 防止除以零
    result = np.clip(result, 1e-15, 1-1e-15)
    
    return result

In [ ]:
def cross_entropy_loss(y, y_pred):
    """
    计算交叉熵损失
    参数:
        y: 真实标签 (n_samples,)
        y_pred: 预测概率 (n_samples,)
    返回:
        平均交叉熵损失
    """
    epsilon = 1e-15  # 防止log(0)出现
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)  # 限制预测值范围
    return -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))

In [ ]:
# ====== 步骤六：编写该梯度的计算程序 ======
def compute_gradients(X, y, y_pred):
    """
    计算损失函数关于权重和偏置的梯度
    参数:
        X: 特征矩阵 (n_samples, n_features)
        y: 真实标签 (n_samples,)
        y_pred: 预测概率 (n_samples,)
    返回:
        dw: 权重梯度 (n_features,)
        db: 偏置梯度 (标量)
    """
    m = len(y)
    error = y_pred - y  # 预测值与真实值的差
    dw = (1/m) * np.dot(X.T, error)  # 权重梯度
    db = (1/m) * np.sum(error)  # 偏置梯度
    return dw, db

In [ ]:
# ====== 步骤七：梯度下降优化函数 ======
def gradient_descent(X, y, learning_rate, n_iters):
    """
    使用梯度下降优化逻辑回归模型
    参数:
        X: 特征矩阵 (n_samples, n_features)
        y: 标签向量 (n_samples,)
        learning_rate: 学习率
        n_iters: 迭代次数
    返回:
        w: 优化后的权重
        b: 优化后的偏置
        losses: 每次迭代的损失值
    """
    m, n = X.shape
    # 初始化权重和偏置
    w = np.zeros(n)
    b = 0.0
    
    losses = []
    
    for i in range(n_iters):
        # 计算预测概率
        linear = np.dot(X, w) + b
        y_pred = sigmoid(linear)
        
        # 计算损失
        loss = cross_entropy_loss(y, y_pred)
        losses.append(loss)
        
        # 计算梯度
        dw, db = compute_gradients(X, y, y_pred)
        
        # 更新参数
        w -= learning_rate * dw
        b -= learning_rate * db
        
        # 每100次迭代打印进度
        if i % 100 == 0:
            print(f"Iteration {i}/{n_iters}, Loss: {loss:.4f}")
    
    print(f"Final loss: {loss:.4f}")
    return w, b, losses

In [ ]:
# ====== 步骤八：加载并可视化数据集1 ======
# 加载数据集1
data1 = np.loadtxt('/mnt/e/平时作业/机器学习/实验1/ex2data1.txt', delimiter=',')
X = data1[:, :2]  # 前两列是特征
y= data1[:, 2]   # 第三列是标签

# 可视化数据集1
plt.figure(figsize=(10, 6))
plt.scatter(X[y==0][:, 0], X[y==0][:, 1], c='blue',  marker='x',label='Not Admitted')
plt.scatter(X[y==1][:, 0], X[y==1][:, 1], c='red', label='Admitted')
plt.xlabel('Exam 1 Score')
plt.ylabel('Exam 2 Score')
plt.title('Dataset 1: Admission Decisions')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ====== 步骤九：添加全1向量 ======
# 添加全1向量作为偏置项
X = np.c_[np.ones(X.shape[0]), X]  # 在特征矩阵前添加一列1

# ====== 数据预处理 ======
# 划分训练集和测试集（80%训练，20%测试）
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 特征标准化
scaler = StandardScaler()
X_train[:, 1:] = scaler.fit_transform(X_train[:, 1:])
X_test[:, 1:] = scaler.transform(X_test[:, 1:])

In [ ]:
# ====== 步骤十：初始化权重 ======
# 在梯度下降函数中已完成初始化

# ====== 步骤十一：梯度下降优化 ======
# 设置超参数
learning_rate = 0.01
n_iters = 1000

# 训练模型
w, b, losses = gradient_descent(X_train, y_train, learning_rate, n_iters)

# ====== 步骤十二：可视化历史损失函数 ======
plt.figure(figsize=(10, 6))
plt.plot(range(n_iters), losses)
plt.title('Training Loss over Iterations')
plt.xlabel('Iteration')
plt.ylabel('Cross-Entropy Loss')
plt.grid(True)
plt.show()

In [ ]:
# ====== 步骤十三：观察迭代次数和学习率的影响 ======
# 测试不同学习率
learning_rates = [0.001, 0.01, 0.1, 1.0]
plt.figure(figsize=(12, 8))

for lr in learning_rates:
    _, _, losses = gradient_descent(X_train, y_train, lr, n_iters)
    plt.plot(range(n_iters), losses, label=f'LR={lr}')

plt.title('Effect of Learning Rate on Training Loss')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 测试不同迭代次数
iterations = [100, 500, 1000, 2000]
plt.figure(figsize=(12, 8))

for iters in iterations:
    _, _, losses = gradient_descent(X_train, y_train, learning_rate, iters)
    plt.plot(range(iters), losses, label=f'Iter={iters}')

plt.title('Effect of Iterations on Training Loss')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ====== 步骤十四：构建测试集并输出预测类别 ======
def predict(X, w, b):
    """
    使用训练好的模型进行预测
    参数:
        X: 特征矩阵
        w: 权重向量
        b: 偏置项
    返回:
        预测类别 (0或1)
    """
    linear = np.dot(X, w) + b
    y_pred_prob = sigmoid(linear)
    return (y_pred_prob >= 0.5).astype(int)

# 在测试集上进行预测
y_pred = predict(X_test, w, b)

# 输出测试样本预测结果
print("\nTest set predictions:")
print("True\tPredicted")
for true, pred in zip(y_test[:10], y_pred[:10]):  # 只打印前10个样本
    print(f"{int(true)}\t{pred}")

# 计算测试集准确率
accuracy = np.mean(y_pred == y_test)
print(f"\nTest set accuracy: {accuracy*100:.2f}%")

In [ ]:
# 可视化决策边界
def plot_decision_boundary(X, y, w, b):
    """
    绘制决策边界
    """
    plt.figure(figsize=(10, 6))
    
    # 绘制原始数据点
    plt.scatter(X[y==0][:, 1], X[y==0][:, 2], c='blue', marker='x', label='Not Admitted')
    plt.scatter(X[y==1][:, 1], X[y==1][:, 2], c='red', marker='o', label='Admitted')
    
    # 生成决策边界
    x1_min, x1_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    x2_min, x2_max = X[:, 2].min() - 1, X[:, 2].max() + 1
    xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 100),
                           np.linspace(x2_min, x2_max, 100))
    
    # 预测网格点的类别
    Z = predict(np.c_[np.ones(len(xx1.ravel())), xx1.ravel(), xx2.ravel()], w, b)
    Z = Z.reshape(xx1.shape)
    
    # 绘制决策边界
    plt.contourf(xx1, xx2, Z, alpha=0.3, cmap='coolwarm')
    plt.xlabel('Exam 1 Score (Standardized)')
    plt.ylabel('Exam 2 Score (Standardized)')
    plt.title('Decision Boundary of Logistic Regression')
    plt.legend()
    plt.grid(True)
    plt.show()

# 绘制决策边界
plot_decision_boundary(X_train, y_train, w, b)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split

In [ ]:
# ====== 步骤十六：载入ex2data1（芯片质量数据集）并可视化 ======
# 加载数据集
data_linear = np.loadtxt('/mnt/e/平时作业/机器学习/实验1/ex2data2.txt', delimiter=',')
X_linear = data_linear[:, :2]  # 前两列是特征
y_linear = data_linear[:, 2]   # 第三列是标签

# 可视化线性不可分数据集
plt.figure(figsize=(10, 6))
plt.scatter(X_linear[y_linear==0][:, 0], X_linear[y_linear==0][:, 1], 
            c='blue', marker='x', label='Rejected')
plt.scatter(X_linear[y_linear==1][:, 0], X_linear[y_linear==1][:, 1], 
            c='red', marker='o', label='Accepted')
plt.xlabel('Microchip Test 1')
plt.ylabel('Microchip Test 2')
plt.title('Microchip Quality Test Results (Linear Inseparable)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ====== 步骤十七：对二维特征进行非线性映射 ======
def map_features(X1, X2, degree=6):
    """
    将二维特征映射到多项式特征空间
    参数:
        X1: 第一维特征向量
        X2: 第二维特征向量
        degree: 多项式次数
    返回:
        多项式特征矩阵
    """
    # 初始化特征矩阵，第一列为1（偏置项）
    m = len(X1)
    features = np.ones((m, 1))
    
    # 生成多项式特征
    for i in range(1, degree+1):
        for j in range(0, i+1):
            # 特征项: x1^(i-j) * x2^j
            new_feature = (X1**(i-j)) * (X2**j)
            features = np.c_[features, new_feature]
    
    return features

# 应用非线性映射（6次多项式）
X1 = X_linear[:, 0]
X2 = X_linear[:, 1]
X_poly = map_features(X1, X2, degree=6)

In [ ]:
# ====== 步骤十八：基于非线性映射后的特征进行对数几率回归 ======
# 重新定义带正则化的损失函数和梯度

def regularized_cross_entropy_loss(X, y, w, lambda_=100.0):
    """
    带正则化的交叉熵损失函数
    参数:
        X: 特征矩阵
        y: 标签向量
        w: 权重向量
        lambda_: 正则化参数
    返回:
        正则化损失值
    """
    m = len(y)
    linear = np.dot(X, w)
    y_pred = sigmoid(linear)
    
    # 计算基础损失
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    loss = -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))
    
    # 添加L2正则化项（注意：通常不正则化偏置项）
    reg_loss = (lambda_ / (2 * m)) * np.sum(w[1:]**2)
    
    return loss + reg_loss


In [ ]:
def regularized_gradients(X, y, w, lambda_=100.0):
    """
    计算带正则化的梯度
    参数:
        X: 特征矩阵
        y: 标签向量
        w: 权重向量
        lambda_: 正则化参数
    返回:
        梯度向量
    """
    m = len(y)
    linear = np.dot(X, w)
    y_pred = sigmoid(linear)
    
    # 计算基础梯度
    error = y_pred - y
    grad = (1 / m) * np.dot(X.T, error)
    
    # 添加正则化梯度（不应用于偏置项）
    reg_grad = (lambda_ / m) * w
    reg_grad[0] = 0  # 不对偏置项正则化
    
    return grad + reg_grad

In [ ]:
# ====== 重写梯度下降函数 - 添加梯度限制防止爆炸 ======
def regularized_gradient_descent(X, y, learning_rate, n_iters, lambda_=100.0):
    """
    带正则化的梯度下降优化 - 修复梯度爆炸问题
    """
    m, n = X.shape
    w = np.zeros(n)  # 初始化权重
    losses = []
    accuracies = []
    
    for i in range(n_iters):
        # 计算预测值
        linear = np.dot(X, w)
        y_pred = sigmoid(linear)
        
        # 计算损失（带正则化）
        loss = regularized_cross_entropy_loss(X, y, w, lambda_)
        losses.append(loss)
        
        # 计算准确率
        accuracy = np.mean((y_pred >= 0.5) == y)
        accuracies.append(accuracy)
        
        # 计算梯度（带正则化）
        grad = regularized_gradients(X, y, w, lambda_)
        
        # 梯度裁剪 - 防止梯度爆炸
        grad_norm = np.linalg.norm(grad)
        max_grad_norm = 1000  # 最大梯度范数阈值
        if grad_norm > max_grad_norm:
            grad = (grad / grad_norm) * max_grad_norm
        
        # 更新权重
        w -= learning_rate * grad
        
        # 每1000次迭代打印进度
        if i % 1000 == 0:
            print(f"Iteration {i}/{n_iters}, Loss: {loss:.4f}, Accuracy: {accuracy:.4f}")
    
    print(f"Final loss: {loss:.4f}, Final accuracy: {accuracies[-1]:.4f}")
    return w, losses, accuracies

In [ ]:
# ====== 步骤二十：分析正则化参数对训练精度的影响 ======
# 准备数据
scaler_poly = StandardScaler()
X_poly_scaled = scaler_poly.fit_transform(X_poly[:, 1:])  # 标准化多项式特征
X_poly_scaled = np.c_[np.ones(X_poly_scaled.shape[0]), X_poly_scaled]  # 添加偏置项

# 划分训练集和测试集
X_train_poly, X_test_poly, y_train_poly, y_test_poly = train_test_split(
    X_poly_scaled, y_linear, test_size=0.2, random_state=42
)

# 测试不同正则化参数
lambda_values = [0, 0.001, 0.01, 0.1, 1, 10, 100]
train_accuracies = []
test_accuracies = []

plt.figure(figsize=(12, 8))

for lambda_val in lambda_values:
    print(f"\nTraining with lambda = {lambda_val}")
    # 训练模型
    w_poly, losses_poly, accuracies_poly = regularized_gradient_descent(
        X_train_poly, y_train_poly, 
        learning_rate=0.1, 
        n_iters=10000, 
        lambda_=lambda_val
    )
    
    # 计算训练集准确率
    train_pred = predict(X_train_poly, w_poly, 0)  # 这里b已包含在w中
    train_acc = np.mean(train_pred == y_train_poly)
    train_accuracies.append(train_acc)
    
    # 计算测试集准确率
    test_pred = predict(X_test_poly, w_poly, 0)
    test_acc = np.mean(test_pred == y_test_poly)
    test_accuracies.append(test_acc)
    
    print(f"Lambda = {lambda_val}: Train Acc = {train_acc:.4f}, Test Acc = {test_acc:.4f}")
    
    # 绘制损失曲线
    plt.plot(losses_poly, label=f"λ={lambda_val}")

plt.title('Training Loss with Different Regularization Parameters')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# 可视化正则化参数对准确率的影响
plt.figure(figsize=(12, 6))
plt.semilogx(lambda_values, train_accuracies, 'bo-', label='Train Accuracy')
plt.semilogx(lambda_values, test_accuracies, 'ro-', label='Test Accuracy')
plt.title('Effect of Regularization Parameter on Accuracy')
plt.xlabel('Regularization Parameter (λ) - Log Scale')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ====== 步骤二十一：编写训练集分类精度的计算公式 ======
def calculate_accuracy(y_true, y_pred):
    """
    计算分类准确率
    参数:
        y_true: 真实标签
        y_pred: 预测标签
    返回:
        准确率（百分比）
    """
    correct = np.sum(y_true == y_pred)
    total = len(y_true)
    return (correct / total) * 100

# 训练集准确率计算示例
print("\n训练集分类精度计算:")
print(f"训练样本数: {len(y_test_poly)}")
print(f"正确分类数: {np.sum(test_pred == y_test_poly)}")
print(f"分类准确率: {test_acc:.4%}")


In [ ]:
# ====== 修正决策边界可视化函数 - 解决DeprecationWarning ======
def plot_decision_boundary_poly(X, y, w, degree=6):
    """
    绘制多项式特征的决策边界 - 修复版
    """
    plt.figure(figsize=(10, 6))
    
    # 绘制原始数据点
    plt.scatter(X[y==0][:, 1], X[y==0][:, 2], c='blue', marker='x', label='Rejected', alpha=0.6)
    plt.scatter(X[y==1][:, 1], X[y==1][:, 2], c='red', marker='o', label='Accepted', alpha=0.6)
    
    # 创建网格点
    h = 0.02  # 步长
    x_min, x_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    y_min, y_max = X[:, 2].min() - 0.5, X[:, 2].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # 获取网格点的预测值
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    
    # 为网格点生成多项式特征
    X1 = grid_points[:, 0]
    X2 = grid_points[:, 1]
    grid_poly = map_features(X1, X2, degree)
    
    # 标准化特征
    grid_poly_scaled = grid_poly.copy()
    grid_poly_scaled[:, 1:] = scaler_poly.transform(grid_poly[:, 1:])
    
    # 预测概率值
    linear = np.dot(grid_poly_scaled, w)
    Z = sigmoid(linear).reshape(xx.shape)
    
    # 绘制决策边界（p=0.5）
    plt.contour(xx, yy, Z, levels=[0.5], colors='green', linewidths=2)
    
    # 绘制概率等高线
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    
    plt.xlabel('Test 1 (Standardized)')
    plt.ylabel('Test 2 (Standardized)')
    plt.title(f'Decision Boundary (λ={lambda_val})')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
best_lambda_idx = np.argmax(test_accuracies)
lambda_val = lambda_values[best_lambda_idx]
lambda_val = 100
print(f"Selected lambda: {lambda_val} based on test accuracy")

# ====== 使用最佳λ值训练模型 ======
print(f"\nTraining with best lambda = {lambda_val}")
w_poly, losses_poly, accuracies_poly = regularized_gradient_descent(
    X_train_poly, y_train_poly, 
    learning_rate=0.01,  # 降低学习率防止数值不稳定
    n_iters=15000,       # 增加迭代次数
    lambda_=lambda_val
)

# 计算最终准确率
train_pred = predict(X_train_poly, w_poly, 0)
test_pred = predict(X_test_poly, w_poly, 0)
train_acc = np.mean(train_pred == y_train_poly)
test_acc = np.mean(test_pred == y_test_poly)

print(f"\nBest Model Performance:")
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# ====== 可视化决策边界（修复版） ======
plot_decision_boundary_poly(X_train_poly, y_train_poly, w_poly)